# FinGPT Hybrid No-Lookahead News Scoring — 85 Jobs

This notebook produces a clean 85-row signal file from:

`/content/finrobot2_rebalanced_panel.csv`

Final output:

`/content/fingpt_hybrid_level_adjusted_signals.csv`

Design:
1. Fetch Finnhub company news using a strict no-lookahead window.
2. Compute deterministic base scores: `sentiment_score`, `confidence_score`, `materiality_score`.
3. Use FinGPT as a qualitative judge only: it decides whether each base score is `too_high`, `too_low`, or `reasonable`.
4. If FinGPT says `too_high`, the score is reduced; if it says `too_low`, the score is increased; if `reasonable`, it is kept.
5. If FinGPT fails, the notebook falls back to deterministic base scores, so the 85-job run still finishes.

Calibration formula:

\[
A(d;\delta)=
egin{cases}
-\delta, & d=	ext{too\_high},\
+\delta, & d=	ext{too\_low},\
0, & d=	ext{reasonable}.
\end{cases}
\]

\[
s=\operatorname{clip}(s_0 + A(d_s;0.10), -1, 1),\quad
c=\operatorname{clip}(c_0 + A(d_c;0.05), 0, 1),\quad
m=\operatorname{clip}(m_0 + A(d_m;0.05), 0, 1).
\]


In [ ]:
# 0. Imports and configuration
import os
import re
import json
import time
import getpass
import importlib
import sys
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd

# Input/output paths. These match your Colab file names.
CSV_PATH = Path('/content/finrobot2_rebalanced_panel.csv')
OUT_CSV = Path('/content/fingpt_hybrid_level_adjusted_signals.csv')

# Core run configuration
N_WEEKS = 3
RUN_UNIQUE_TICKER_DATE = True
SLEEP_BETWEEN_CALLS = 1
OVERWRITE_OUTPUT = True

# Hybrid switch: True = use FinGPT as qualitative judge if model loads successfully.
# If loading/app generation fails, the run automatically falls back to deterministic keyword score.
USE_FINGPT_JUDGE = True

# FinGPT source path in your Colab environment.
FORECASTER_DIR = Path('/content/FinGPT-src/fingpt/FinGPT_Forecaster')
APP_PATH = FORECASTER_DIR / 'app.py'

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('CSV_PATH:', CSV_PATH)
print('CSV exists:', CSV_PATH.exists())
print('APP_PATH:', APP_PATH)
print('app.py exists:', APP_PATH.exists())
print('OUT_CSV:', OUT_CSV)

assert CSV_PATH.exists(), f'Cannot find input CSV at {CSV_PATH}. Please upload finrobot2_rebalanced_panel.csv to /content.'


In [ ]:
# 1. Finnhub key setup
# You only need FINNHUB_API_KEY for news retrieval.
# HF_TOKEN is not needed for deterministic scoring, but FinGPT model loading may require it depending on your local setup/model.

if not os.environ.get('FINNHUB_API_KEY'):
    os.environ['FINNHUB_API_KEY'] = getpass.getpass('FINNHUB_API_KEY: ')

os.environ['FINNHUB_KEY'] = os.environ['FINNHUB_API_KEY']

try:
    import finnhub
except ImportError:
    raise ImportError('Please run: !pip install finnhub-python')

finnhub_client = finnhub.Client(api_key=os.environ['FINNHUB_API_KEY'])
print('Finnhub client ready.')


In [ ]:
# 2. Load FinGPT app only if needed
# This can be slow and GPU-heavy. If it fails, the notebook will still run with deterministic fallback.

app = None
FINGPT_AVAILABLE = False

if USE_FINGPT_JUDGE:
    try:
        assert APP_PATH.exists(), f'Cannot find app.py at {APP_PATH}'
        if str(FORECASTER_DIR) not in sys.path:
            sys.path.insert(0, str(FORECASTER_DIR))
        import app as _app
        app = importlib.reload(_app)
        FINGPT_AVAILABLE = hasattr(app, 'tokenizer') and hasattr(app, 'model') and hasattr(app, 'torch')
        print('FinGPT app loaded:', FINGPT_AVAILABLE)
    except Exception as exc:
        print('FinGPT app loading failed. Will use deterministic fallback only.')
        print('Error:', repr(exc))
        app = None
        FINGPT_AVAILABLE = False
else:
    print('USE_FINGPT_JUDGE = False. Will use deterministic scoring only.')


In [ ]:
# 3. Ticker-specific company terms for news filtering
company_terms = {
    'AMAT': ['amat', 'applied materials', 'semiconductor equipment', 'wafer', 'fab', 'chip equipment', 'deposition', 'etch'],
    'AMD': ['amd', 'advanced micro devices', 'mi300', 'mi350', 'gpu', 'cpu', 'data center', 'lisa su', 'epyc', 'instinct'],
    'AMZN': ['amazon', 'aws', 'prime', 'e-commerce', 'andy jassy', 'cloud', 'advertising', 'retail'],
    'COST': ['costco', 'membership', 'same-store sales', 'comparable sales', 'retail', 'warehouse', 'traffic'],
    'CSCO': ['csco', 'cisco', 'networking', 'switches', 'routers', 'security', 'webex', 'enterprise networking'],
    'GOOGL': ['google', 'alphabet', 'youtube', 'search', 'gemini', 'waymo', 'cloud', 'google cloud', 'sundar pichai'],
    'GOOG': ['google', 'alphabet', 'youtube', 'search', 'gemini', 'waymo', 'cloud', 'google cloud', 'sundar pichai'],
    'ISRG': ['intuitive surgical', 'isrg', 'da vinci', 'robotic surgery', 'surgical systems', 'procedures'],
    'META': ['meta', 'facebook', 'instagram', 'whatsapp', 'threads', 'reality labs', 'mark zuckerberg', 'advertising'],
    'NFLX': ['netflix', 'streaming', 'subscribers', 'content', 'ads', 'password sharing', 'paid sharing'],
    'TSLA': ['tesla', 'elon musk', 'ev', 'model y', 'model 3', 'cybertruck', 'fsd', 'robotaxi', 'energy storage'],
    'TXN': ['txn', 'texas instruments', 'analog chips', 'embedded processing', 'semiconductor', 'industrial chips'],
}

positive_keywords = {
    'beat': 0.16,
    'beats': 0.16,
    'strong': 0.10,
    'growth': 0.10,
    'upgrade': 0.14,
    'raises': 0.12,
    'raised': 0.12,
    'record': 0.10,
    'surge': 0.10,
    'rally': 0.08,
    'expansion': 0.08,
    'partnership': 0.08,
    'exceeded expectations': 0.14,
    'bestselling': 0.10,
    'best-selling': 0.10,
    'launch': 0.08,
    'profit': 0.08,
    'cloud growth': 0.12,
    'ai comeback': 0.10,
    'strengthening': 0.10,
    'improves': 0.10,
    'improved': 0.10,
}

negative_keywords = {
    'miss': 0.16,
    'misses': 0.16,
    'weak': 0.14,
    'weakening': 0.16,
    'slowdown': 0.14,
    'downgrade': 0.16,
    'cut': 0.12,
    'cuts': 0.12,
    'delay': 0.12,
    'delays': 0.12,
    'risk': 0.10,
    'concerns': 0.12,
    'concern': 0.12,
    'dipped': 0.10,
    'fell': 0.10,
    'falls': 0.10,
    'decline': 0.10,
    'sell': 0.10,
    'strong sell': 0.22,
    'competition': 0.12,
    'challenging': 0.12,
    'revised down': 0.20,
    'weighing on': 0.12,
    'uncertain': 0.10,
    'lawsuit': 0.14,
    'probe': 0.14,
    'investigation': 0.14,
    'slow': 0.10,
}

neg_phrases = ['weakening demand', 'weak demand', 'demand concerns', 'demand slowdown']
pos_phrases = ['strong demand', 'robust demand', 'surging demand']


In [ ]:
# 4. Date normalization and no-lookahead news retrieval

def normalize_date(x):
    s = str(x).strip()
    if not s or s.lower() == 'nan':
        return None

    # If date is like 01/02, assume year 2026.
    if '/' in s:
        parts = s.split('/')
        if len(parts) == 2:
            s = f'2026/{s}'

    dt = pd.to_datetime(s, errors='coerce')
    if pd.isna(dt):
        raise ValueError(f'Cannot parse date: {x}')
    return dt.strftime('%Y-%m-%d')


def get_news_window(rebalance_date, n_weeks=3):
    """Return [start_date, end_date], where end_date is one day before rebalance_date."""
    end_dt = datetime.strptime(rebalance_date, '%Y-%m-%d') - timedelta(days=1)
    start_dt = end_dt - timedelta(days=7 * n_weeks)
    return start_dt.strftime('%Y-%m-%d'), end_dt.strftime('%Y-%m-%d')


def fetch_no_lookahead_news(ticker, rebalance_date, n_weeks=3):
    """Fetch and filter Finnhub company news strictly before rebalance_date."""
    ticker = ticker.upper().strip()
    start_date, end_date = get_news_window(rebalance_date, n_weeks)
    cutoff_dt = datetime.strptime(rebalance_date, '%Y-%m-%d').date()
    terms = company_terms.get(ticker, [ticker.lower()])

    try:
        raw_news = finnhub_client.company_news(ticker, _from=start_date, to=end_date)
    except Exception as exc:
        return [], start_date, end_date, repr(exc)

    filtered = []
    seen = set()

    for item in raw_news:
        pub_ts = item.get('datetime')
        if not pub_ts:
            continue

        pub_dt = datetime.fromtimestamp(pub_ts)
        if pub_dt.date() >= cutoff_dt:
            continue

        headline = (item.get('headline') or '').strip()
        summary = (item.get('summary') or '').strip()
        blob = f'{headline} {summary}'.lower()

        if not headline and not summary:
            continue
        if summary.startswith('Looking for stock market analysis and research with proves results?'):
            continue
        if not any(term in blob for term in terms):
            continue

        key = (headline.lower(), summary[:160].lower())
        if key in seen:
            continue
        seen.add(key)

        filtered.append({
            'date': pub_dt.strftime('%Y-%m-%d'),
            'headline': headline,
            'summary': summary,
        })

    filtered.sort(key=lambda x: x['date'])
    return filtered, start_date, end_date, None


In [ ]:
# 5. Deterministic base scoring from no-lookahead news

def score_from_news_items(news_items):
    text = "\n".join(
        f"{x.get('headline', '')}. {x.get('summary', '')}" for x in news_items
    )
    lower = text.lower()

    raw_score = 0.0
    pos_hits = []
    neg_hits = []

    for phrase in pos_phrases:
        if phrase in lower:
            raw_score += 0.16
            pos_hits.append(phrase)

    for phrase in neg_phrases:
        if phrase in lower:
            raw_score -= 0.18
            neg_hits.append(phrase)

    for kw, weight in positive_keywords.items():
        if kw in lower:
            if kw == 'demand' and any(p in lower for p in neg_phrases):
                continue
            raw_score += weight
            pos_hits.append(kw)

    for kw, weight in negative_keywords.items():
        if kw in lower:
            raw_score -= weight
            neg_hits.append(kw)

    n_news = len(news_items)

    if n_news > 0 and abs(raw_score) < 1e-9:
        raw_score = 0.03

    sentiment_score = max(-0.60, min(0.60, raw_score))
    confidence_score = min(0.75, max(0.45, 0.40 + 0.03 * min(n_news, 10)))
    materiality_score = min(0.80, max(0.25, 0.20 + 0.05 * min(n_news, 12)))

    return {
        'sentiment_score': round(float(sentiment_score), 4),
        'confidence_score': round(float(confidence_score), 4),
        'materiality_score': round(float(materiality_score), 4),
        'n_news': n_news,
        'positive_hits': '; '.join(pos_hits[:8]),
        'negative_hits': '; '.join(neg_hits[:8]),
    }


In [ ]:
# 6. FinGPT judge and score-level adjustment rule

def finGPT_judge_score_level(ticker, rebalance_date, news_items, base_scores):
    """
    FinGPT does not output final scores or JSON.
    It only judges whether each deterministic base score is:
    too_high / too_low / reasonable.
    """
    if not FINGPT_AVAILABLE or app is None:
        raise RuntimeError('FinGPT app/model is not available.')

    news_text = "\n".join(
        f"- {x.get('date', '')}: {x.get('headline', '')}. {x.get('summary', '')}"
        for x in news_items[:12]
    )

    if not news_text.strip():
        news_text = 'No company-specific news found before the rebalance date.'

    prompt = f"""[INST]<<SYS>>
You are a financial news score calibration judge.

You are NOT asked to generate JSON.
You are NOT asked to write a report.
You are NOT asked to produce final numeric scores.

You will receive:
1. No-look-ahead company news.
2. Deterministic base scores.

Your only task is to judge whether each base score is too high, too low, or reasonable.

Use exactly one of these labels for each field:
too_high
too_low
reasonable

Definitions:
- sentiment_score too_high means the base sentiment is too bullish / too positive relative to the news.
- sentiment_score too_low means the base sentiment is too bearish / too negative relative to the news.
- confidence_score too_high means the base confidence is too strong given the clarity of news.
- confidence_score too_low means the base confidence is too weak given the clarity of news.
- materiality_score too_high means the base materiality overstates the importance of the news.
- materiality_score too_low means the base materiality understates the importance of the news.

Return exactly this format and nothing else:

sentiment_score: too_high/too_low/reasonable
confidence_score: too_high/too_low/reasonable
materiality_score: too_high/too_low/reasonable
<</SYS>>

Ticker: {ticker}
Rebalance date: {rebalance_date}

Base scores:
sentiment_score: {base_scores['sentiment_score']}
confidence_score: {base_scores['confidence_score']}
materiality_score: {base_scores['materiality_score']}

No-look-ahead news:
{news_text}

Return exactly:
sentiment_score: too_high/too_low/reasonable
confidence_score: too_high/too_low/reasonable
materiality_score: too_high/too_low/reasonable
[/INST]"""

    inputs = app.tokenizer(prompt, return_tensors='pt', padding=False)
    inputs = {k: v.to(app.model.device) for k, v in inputs.items()}

    with app.torch.no_grad():
        res = app.model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            eos_token_id=app.tokenizer.eos_token_id,
            pad_token_id=app.tokenizer.eos_token_id,
            use_cache=True,
        )

    output = app.tokenizer.decode(res[0], skip_special_tokens=True)
    answer = re.sub(r'.*\[/INST\]\s*', '', output, flags=re.DOTALL).strip()

    return answer


def apply_fingpt_score_level_adjustment(
    base_scores,
    judge_text,
    sentiment_delta=0.10,
    confidence_delta=0.05,
    materiality_delta=0.05,
):
    """
    Rule:
    - If FinGPT says too_high, decrease the score.
    - If FinGPT says too_low, increase the score.
    - If FinGPT says reasonable or parsing fails, keep it unchanged.
    """
    s = float(base_scores['sentiment_score'])
    c = float(base_scores['confidence_score'])
    m = float(base_scores['materiality_score'])

    text = str(judge_text).lower()

    def extract_label(field):
        pattern = rf'{field}\s*:\s*(too_high|too_low|reasonable)'
        match = re.search(pattern, text)
        if match:
            return match.group(1)
        return 'reasonable'

    sentiment_label = extract_label('sentiment_score')
    confidence_label = extract_label('confidence_score')
    materiality_label = extract_label('materiality_score')

    def adjustment(label, delta):
        if label == 'too_high':
            return -delta
        if label == 'too_low':
            return delta
        return 0.0

    s = s + adjustment(sentiment_label, sentiment_delta)
    c = c + adjustment(confidence_label, confidence_delta)
    m = m + adjustment(materiality_label, materiality_delta)

    s = max(-1.0, min(1.0, s))
    c = max(0.0, min(1.0, c))
    m = max(0.0, min(1.0, m))

    return {
        'sentiment_score': round(s, 4),
        'confidence_score': round(c, 4),
        'materiality_score': round(m, 4),
        'fingpt_judge_text': judge_text,
        'fingpt_sentiment_label': sentiment_label,
        'fingpt_confidence_label': confidence_label,
        'fingpt_materiality_label': materiality_label,
    }


In [ ]:
# 7. Load 85 ticker-date jobs

df = pd.read_csv(CSV_PATH)
print('CSV columns:', list(df.columns))
print('CSV shape:', df.shape)


def find_col(candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    raise ValueError(f'Cannot find column among {candidates}. Existing columns: {list(df.columns)}')


ticker_col = find_col(['ticker', 'symbol'])
date_col = find_col(['rebalance_date', 'date', 'trade_date'])

jobs = df[[ticker_col, date_col]].dropna().copy()
jobs['ticker'] = jobs[ticker_col].astype(str).str.upper().str.strip()
jobs['rebalance_date'] = jobs[date_col].apply(normalize_date)
jobs = jobs[['ticker', 'rebalance_date']].dropna()

if RUN_UNIQUE_TICKER_DATE:
    jobs = jobs.drop_duplicates()

print('Total jobs:', len(jobs))
jobs.head()


In [ ]:
# 8. Optional: delete old output to avoid mixing versions
if OVERWRITE_OUTPUT and OUT_CSV.exists():
    OUT_CSV.unlink()
    print('Deleted old output:', OUT_CSV)
else:
    print('No old output deleted.')


In [ ]:
# 9. Run all jobs and save incrementally

completed = set()

if OUT_CSV.exists() and not OVERWRITE_OUTPUT:
    old = pd.read_csv(OUT_CSV)
    if {'ticker', 'rebalance_date'}.issubset(old.columns):
        completed = set(zip(old['ticker'], old['rebalance_date']))

print(f'Already completed: {len(completed)}')
print(f'Output: {OUT_CSV}')
print(f'FinGPT available for judging: {FINGPT_AVAILABLE}')

rows = []

for i, row in jobs.reset_index(drop=True).iterrows():
    ticker = row['ticker']
    rebalance_date = row['rebalance_date']
    key = (ticker, rebalance_date)

    if key in completed:
        print(f'[skip] {i+1}/{len(jobs)} {ticker} {rebalance_date}')
        continue

    print(f'[run] {i+1}/{len(jobs)} {ticker} {rebalance_date}')

    result = {
        'ticker': ticker,
        'rebalance_date': rebalance_date,
        'status': 'ok',
    }

    try:
        # Step 1: Fetch no-lookahead news
        news_items, window_start, window_end, fetch_error = fetch_no_lookahead_news(
            ticker,
            rebalance_date,
            N_WEEKS,
        )

        result['news_window_start'] = window_start
        result['news_window_end'] = window_end

        if fetch_error is not None:
            result['status'] = 'news_fetch_error'
            result['error'] = fetch_error

            base_scores = {
                'sentiment_score': 0.0,
                'confidence_score': 0.0,
                'materiality_score': 0.0,
                'n_news': 0,
                'positive_hits': '',
                'negative_hits': '',
            }

            result.update(base_scores)
            result['base_sentiment_score'] = 0.0
            result['base_confidence_score'] = 0.0
            result['base_materiality_score'] = 0.0
            result['fingpt_judge_text'] = ''
            result['fingpt_sentiment_label'] = 'not_run'
            result['fingpt_confidence_label'] = 'not_run'
            result['fingpt_materiality_label'] = 'not_run'
            result['source'] = 'news_fetch_error_zero_score'

        else:
            # Step 2: Deterministic base score
            base_scores = score_from_news_items(news_items)

            result['base_sentiment_score'] = base_scores['sentiment_score']
            result['base_confidence_score'] = base_scores['confidence_score']
            result['base_materiality_score'] = base_scores['materiality_score']
            result['n_news'] = base_scores.get('n_news', len(news_items))
            result['positive_hits'] = base_scores.get('positive_hits', '')
            result['negative_hits'] = base_scores.get('negative_hits', '')

            # Step 3: FinGPT judge says whether base score is too high / too low / reasonable
            try:
                judge_text = finGPT_judge_score_level(
                    ticker=ticker,
                    rebalance_date=rebalance_date,
                    news_items=news_items,
                    base_scores=base_scores,
                )

                adjusted_scores = apply_fingpt_score_level_adjustment(
                    base_scores=base_scores,
                    judge_text=judge_text,
                    sentiment_delta=0.10,
                    confidence_delta=0.05,
                    materiality_delta=0.05,
                )

                result['sentiment_score'] = adjusted_scores['sentiment_score']
                result['confidence_score'] = adjusted_scores['confidence_score']
                result['materiality_score'] = adjusted_scores['materiality_score']

                result['fingpt_judge_text'] = adjusted_scores['fingpt_judge_text']
                result['fingpt_sentiment_label'] = adjusted_scores['fingpt_sentiment_label']
                result['fingpt_confidence_label'] = adjusted_scores['fingpt_confidence_label']
                result['fingpt_materiality_label'] = adjusted_scores['fingpt_materiality_label']

                result['source'] = 'hybrid_fingpt_level_adjusted_score'

            except Exception as judge_exc:
                # If FinGPT judge fails, keep deterministic base scores.
                result['sentiment_score'] = base_scores['sentiment_score']
                result['confidence_score'] = base_scores['confidence_score']
                result['materiality_score'] = base_scores['materiality_score']

                result['fingpt_error'] = repr(judge_exc)
                result['fingpt_judge_text'] = ''
                result['fingpt_sentiment_label'] = 'fallback_reasonable'
                result['fingpt_confidence_label'] = 'fallback_reasonable'
                result['fingpt_materiality_label'] = 'fallback_reasonable'

                result['source'] = 'fallback_news_keyword_score'

        # Step 4: Final composite signal
        result['composite_signal'] = (
            float(result.get('sentiment_score', 0.0))
            * float(result.get('confidence_score', 0.0))
            * float(result.get('materiality_score', 0.0))
        )

    except Exception as exc:
        result['status'] = 'error'
        result['error'] = repr(exc)
        print(f'[error] {ticker} {rebalance_date}: {repr(exc)}')

    rows.append(result)

    # Save incrementally after every job
    new_df = pd.DataFrame(rows)

    if OUT_CSV.exists():
        old = pd.read_csv(OUT_CSV)
        new_df = pd.concat([old, new_df], ignore_index=True)

    new_df.to_csv(OUT_CSV, index=False)
    rows = []

    time.sleep(SLEEP_BETWEEN_CALLS)

print('batch done')
print(f'saved to: {OUT_CSV}')


In [ ]:
# 10. Validate output
out = pd.read_csv(OUT_CSV)
print('Output shape:', out.shape)
print('\nSource counts:')
print(out['source'].value_counts(dropna=False))
print('\nStatus counts:')
print(out['status'].value_counts(dropna=False))

score_cols = ['sentiment_score', 'confidence_score', 'materiality_score', 'composite_signal']
print('\nScore summary:')
print(out[score_cols].describe())

all_zero_rows = (
    (out['sentiment_score'] == 0)
    & (out['confidence_score'] == 0)
    & (out['materiality_score'] == 0)
).sum()
print('\nAll-zero rows:', all_zero_rows)

out[[
    'ticker',
    'rebalance_date',
    'base_sentiment_score',
    'sentiment_score',
    'base_confidence_score',
    'confidence_score',
    'base_materiality_score',
    'materiality_score',
    'fingpt_sentiment_label',
    'fingpt_confidence_label',
    'fingpt_materiality_label',
    'source',
    'composite_signal',
]].head(20)


In [ ]:
# 11. Download final CSV from Colab
# Uncomment this cell in Colab if you want to download the result.

# from google.colab import files
# files.download(str(OUT_CSV))
